# Verifikasi & Validasi Data Telemetri Testbed ClarkNet

Notebook ini berfungsi untuk melakukan pencocokan (*validation check*) antara beban kerja asli dari **ClarkNet Dataset** (yang disimulasikan oleh generator beban) dengan data metrik riil yang terekam di **Prometheus** dan disimpan dalam berkas `collected_metrics.csv`.

Metodologi ini digunakan untuk membuktikan keabsahan (*validity*) data latih sebelum disuplai ke model LSTM/GRU untuk auto-scaling proactive.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 1. Load Data
df_collected = pd.read_csv('collected_metrics.csv')
df_original = pd.read_csv('dataset/aggregated_clarknet_rps.csv')

print(f"Data hasil ekstraksi Prometheus: {len(df_collected)} baris")
print(f"Data dataset ClarkNet asli: {len(df_original)} baris")

## 2. Hitung Statistik & Akurasi Pengiriman Beban

Kita bandingkan total request yang dikirimkan vs total request yang tersimpan secara kumulatif.

In [ ]:
# Kita asumsikan pengujian memakai jendela waktu (misalnya 15 menit = 900 detik)
duration = len(df_collected)

# Jika pengujian menggunakan offset (seperti trafik puncak index 401022)
# Silakan ubah offset di bawah ini sesuai start index saat pengumpulan data
start_offset = 401022 if duration == 300 else 0

orig_slice = df_original.iloc[start_offset:start_offset + duration].copy()

total_media_sent = orig_slice['Media_Service'].sum()
total_media_rec = df_collected['rps_media'].sum()

total_content_sent = orig_slice['Content_Service'].sum()
total_content_rec = df_collected['rps_content'].sum()

accuracy_media = (1 - abs(total_media_sent - total_media_rec) / total_media_sent) * 100
accuracy_content = (1 - abs(total_content_sent - total_content_rec) / total_content_sent) * 100

print("=== ANALISIS AKURASI TRAFIK ===")
print(f"Media Service - Sent: {total_media_sent} | Recorded: {total_media_rec:.2f} | Akurasi: {accuracy_media:.2f}%")
print(f"Content Service - Sent: {total_content_sent} | Recorded: {total_content_rec:.2f} | Akurasi: {accuracy_content:.2f}%")

## 3. Visualisasi Grafik Perbandingan RPS (Actual vs Dataset)

In [ ]:
plt.figure(figsize=(15, 6))
plt.plot(orig_slice['Media_Service'].values, label='Media RPS (Dataset Asli)', color='emerald', alpha=0.5, linestyle='--')
plt.plot(df_collected['rps_media'].values, label='Media RPS (Recorded Prometheus)', color='teal')

plt.title('Validasi Pengiriman Beban Kerja: Media Service', fontsize=14)
plt.xlabel('Detik ke-X', fontsize=12)
plt.ylabel('Request Per Second (RPS)', fontsize=12)
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

plt.figure(figsize=(15, 6))
plt.plot(orig_slice['Content_Service'].values, label='Content RPS (Dataset Asli)', color='coral', alpha=0.5, linestyle='--')
plt.plot(df_collected['rps_content'].values, label='Content RPS (Recorded Prometheus)', color='indigo')

plt.title('Validasi Pengiriman Beban Kerja: Content Service', fontsize=14)
plt.xlabel('Detik ke-X', fontsize=12)
plt.ylabel('Request Per Second (RPS)', fontsize=12)
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

## 4. Visualisasi Penggunaan CPU & RAM vs Lonjakan RPS

Membuktikan adanya korelasi positif antara peningkatan trafik dengan pemakaian resource kontainer.

In [ ]:
fig, ax1 = plt.subplots(figsize=(15, 6))

color = 'tab:blue'
ax1.set_xlabel('Detik ke-X')
ax1.set_ylabel('Total RPS (Media + Content)', color=color)
total_rps = df_collected['rps_media'] + df_collected['rps_content']
ax1.plot(total_rps.values, color=color, alpha=0.7, label='RPS Gabungan')
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('CPU Usage (%)', color=color)
ax2.plot(df_collected['cpu_media'].values, color='green', alpha=0.8, label='CPU Media')
ax2.plot(df_collected['cpu_content'].values, color='red', alpha=0.8, label='CPU Content')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Korelasi Beban Kerja (RPS) terhadap Penggunaan CPU Kontainer', fontsize=14)
fig.tight_layout()
plt.grid(True, linestyle=':', alpha=0.4)
plt.show()